# 7.5.2 Primärverband nach Wirkprinzip, Einordnung gegen Base Rate

**Zentrale Behauptung**: Der Performance-Abfall bei den Modellen auf der exakten Produktebene betrifft primär die konkrete Produktvariante (z. B. Größen-, Klebe- oder Silber-Variante), nicht das zugrundeliegende **Wirkprinzip (Produktfamilie)**. Auf der gruppierten Ebene übertreffen die Modelle die Majority Baseline (Base Rate) deutlich.

---

## Übersicht der Auswertungen:
1. **Vollständige Tabelle (gruppierte Ebene)**: Macro-F1 und Exact Match für Random, Majority, Zero-Shot, Few-Shot, 2-Stage CoT und Inter-Rater getrennt gegen Experte 1 und Experte 2.
2. **Base Rate der häufigsten Produktfamilie**: Identifikation und Frequenz der häufigsten L&R-Produktfamilie je Experte.
3. **Häufigkeitsverteilung über die 8 Kernfamilien**: Gegenüberstellung der Frequenzen in GT1, GT2, Zero-Shot, Few-Shot und 2-Stage CoT.
4. **Delta (Modell minus Majority-Baseline)**: Explizite Gegenüberstellung der Gewinne auf der gruppierten Ebene (gefordert in 7.4.2).
5. **Empfehlungen außerhalb der 8 Kernfamilien**: Anteil und Aufschlüsselung seltener bzw. nicht-abgedeckter Produktfamilien.



In [1]:
import sys, os, re
import numpy as np
import pandas as pd
from collections import Counter
import random

sys.path.insert(0, os.path.abspath('../analyse_categories'))
sys.path.insert(0, os.path.abspath('../utils_notebook/LR_utils_notebook'))
sys.path.insert(0, os.path.abspath('../utils_notebook'))

from category_loader import PRODUCT_FAMILY_MAP, map_product_set, load_csv, to_clean_set, normalize_image_id, evaluate_product_match
from metrics import best_path_f1

BASE_DATA_DIR = os.path.abspath('../../data/llm_outputs')
GT_DIR = os.path.abspath('../../data/ground_truth/lohmann_rauscher')

df_gt1 = load_csv(os.path.join(GT_DIR, 'Experte1_LR_GroundTruth_normalised.csv'))
df_gt2 = load_csv(os.path.join(GT_DIR, 'Experte2_LR_GroundTruth_normalised.csv'))
df_zero = load_csv(os.path.join(BASE_DATA_DIR, 'zero_shot_lr', 'zero_shot_lr_normalised.csv'))
df_few = load_csv(os.path.join(BASE_DATA_DIR, 'few_shot_lr', 'few_shot_lr_normalised.csv'))
df_two = load_csv(os.path.join(BASE_DATA_DIR, 'two_stage_lr', 'two_stage_lr_normalised.csv'))

for df in [df_gt1, df_gt2, df_zero, df_few, df_two]:
    if not df.empty and 'image_id' in df.columns:
        df['image_id'] = df['image_id'].apply(normalize_image_id)

common_58 = sorted(list(set(df_few['image_id'].dropna()).intersection(set(df_zero['image_id'].dropna())).intersection(set(df_two['image_id'].dropna())).intersection(set(df_gt2['image_id'].dropna()))))
print(f"Auswertung auf der gemeinsamen Schnittmenge: N = {len(common_58)} Wundbilder")



Auswertung auf der gemeinsamen Schnittmenge: N = 58 Wundbilder


## 1. Vollständige Tabelle (Gruppierte Ebene / Produktfamilie)
Macro-F1 (Best-Path Set-F1) und Exact Match getrennt gegen Experte 1 und Experte 2.



In [2]:
catalog_prods = [
    'suprasorb a', 'suprasorb a pro', 'suprasorb a + ag',
    'suprasorb p', 'suprasorb p sensitive', 'suprasorb p sensiflex', 'suprasorb p + phmb',
    'suprasorb x', 'suprasorb x pro', 'suprasorb x + phmb',
    'suprasorb liquacel pro', 'vliwasorb pro', 'vliwasorb sensitive', 'vliwazell pro',
    'solvaline n', 'lomatuell pro', 'vliwaktiv', 'vliwaktiv ag', 'suprasorb cnp'
]

rng = random.Random(42)
rand_f1_e1, rand_em_e1, rand_f1_e2, rand_em_e2 = [], [], [], []
for _ in range(100):
    f1_1, em_1, f1_2, em_2 = [], [], [], []
    for img_id in common_58:
        g1 = df_gt1[df_gt1['image_id'] == img_id].iloc[0]
        g2 = df_gt2[df_gt2['image_id'] == img_id].iloc[0]
        rp, ra = [rng.choice(catalog_prods)], [rng.choice(catalog_prods)]
        
        f1, em = evaluate_product_match(rp, ra, g1.get('praeferenz_produkt'), g1.get('alternative_produkt'), group_by_family=True)
        f1_1.append(f1); em_1.append(em)
        f1, em = evaluate_product_match(rp, ra, g2.get('praeferenz_produkt'), g2.get('alternative_produkt'), group_by_family=True)
        f1_2.append(f1); em_2.append(em)
    rand_f1_e1.append(np.mean(f1_1)); rand_em_e1.append(np.mean(em_1))
    rand_f1_e2.append(np.mean(f1_2)); rand_em_e2.append(np.mean(em_2))

maj_prod = 'suprasorb p'

def eval_model(df_model, col_p, col_a, df_gt):
    f1_list, em_list = [], []
    for img_id in common_58:
        gt_row = df_gt[df_gt['image_id'] == img_id].iloc[0]
        if df_model is None:
            p_val, a_val = maj_prod, ''
        else:
            m_row = df_model[df_model['image_id'] == img_id].iloc[0]
            p_val, a_val = m_row.get(col_p), m_row.get(col_a)
            
        f1, em = evaluate_product_match(p_val, a_val, gt_row.get('praeferenz_produkt'), gt_row.get('alternative_produkt'), group_by_family=True)
        f1_list.append(f1)
        em_list.append(em)
    return np.mean(f1_list), np.mean(em_list)

f1_inter_e1, em_inter_e1 = [], []
f1_inter_e2, em_inter_e2 = [], []
for img_id in common_58:
    g1 = df_gt1[df_gt1['image_id'] == img_id].iloc[0]
    g2 = df_gt2[df_gt2['image_id'] == img_id].iloc[0]
    f1, em = evaluate_product_match(g1.get('praeferenz_produkt'), g1.get('alternative_produkt'), g2.get('praeferenz_produkt'), g2.get('alternative_produkt'), group_by_family=True)
    f1_inter_e2.append(f1); em_inter_e2.append(em)
    f1_inter_e1.append(f1); em_inter_e1.append(em)

models = [
    ('Random Baseline', None, None, None),
    ('Majority Baseline (Suprasorb P)', None, None, None),
    ('Zero-Shot', df_zero, 'praeferenz_wundauflage', 'alternativ_wundauflage'),
    ('Few-Shot', df_few, 'praeferenz_wundauflage', 'alternativ_wundauflage'),
    ('2-Stage CoT', df_two, 'praeferenz_wundauflage', 'alternativ_wundauflage'),
    ('Inter-Rater (E1 vs E2)', 'inter', None, None)
]

rows = []
for name, df_m, col_p, col_a in models:
    if name == 'Random Baseline':
        f1_e1, em_e1 = np.mean(rand_f1_e1), np.mean(rand_em_e1)
        f1_e2, em_e2 = np.mean(rand_f1_e2), np.mean(rand_em_e2)
    elif name == 'Inter-Rater (E1 vs E2)':
        f1_e1, em_e1 = np.mean(f1_inter_e1), np.mean(em_inter_e1)
        f1_e2, em_e2 = np.mean(f1_inter_e2), np.mean(em_inter_e2)
    else:
        f1_e1, em_e1 = eval_model(df_m, col_p, col_a, df_gt1)
        f1_e2, em_e2 = eval_model(df_m, col_p, col_a, df_gt2)
        
    rows.append({
        'Ansatz / Baseline': name,
        'Exp 1 Macro-F1': f'{f1_e1*100:.2f}%',
        'Exp 1 Exact Match': f'{em_e1*100:.2f}%',
        'Exp 2 Macro-F1': f'{f1_e2*100:.2f}%',
        'Exp 2 Exact Match': f'{em_e2*100:.2f}%'
    })

df_tbl_grouped = pd.DataFrame(rows)
display(df_tbl_grouped)



,Ansatz / Baseline,Exp 1 Macro-F1,Exp 1 Exact Match,Exp 2 Macro-F1,Exp 2 Exact Match
0,Random Baseline,26.47%,12.05%,44.82%,10.98%
1,Majority Baseline (Suprasorb P),41.09%,22.41%,56.03%,20.69%
2,Zero-Shot,35.34%,5.17%,57.99%,24.14%
3,Few-Shot,38.74%,6.90%,60.11%,25.86%
4,2-Stage CoT,45.11%,6.90%,65.17%,32.76%
5,Inter-Rater (E1 vs E2),56.26%,25.86%,56.26%,25.86%


## 2. Base Rate der häufigsten Produktfamilie je Experte
Ermittlung der häufigsten Produktfamilie (Majority Class) in GT1 und GT2.



In [3]:
def get_fam_counts(df, col_p):
    fams = []
    for img_id in common_58:
        sub = df[df['image_id'] == img_id]
        if sub.empty: continue
        val_p = sub.iloc[0].get(col_p)
        mapped = map_product_set(to_clean_set(val_p), group_by_family=True)
        fams.extend(list(mapped))
    return Counter(fams)

c_gt1 = get_fam_counts(df_gt1, 'praeferenz_produkt')
c_gt2 = get_fam_counts(df_gt2, 'praeferenz_produkt')

maj_gt1_fam, maj_gt1_count = c_gt1.most_common(1)[0]
maj_gt2_fam, maj_gt2_count = c_gt2.most_common(1)[0]

print(f"Häufigste Produktfamilie GT Experte 1: {maj_gt1_fam} -> {maj_gt1_count}/{len(common_58)} Bilder ({maj_gt1_count/len(common_58)*100:.1f}%)")
print(f"Häufigste Produktfamilie GT Experte 2: {maj_gt2_fam} -> {maj_gt2_count}/{len(common_58)} Bilder ({maj_gt2_count/len(common_58)*100:.1f}%)")



Häufigste Produktfamilie GT Experte 1: Suprasorb P (Schaumstoff) -> 24/58 Bilder (41.4%)
Häufigste Produktfamilie GT Experte 2: Suprasorb P (Schaumstoff) -> 30/58 Bilder (51.7%)


## 3. Häufigkeitsverteilung über die 8 Kernfamilien
Vergleich der Häufigkeiten in GT1, GT2, Zero-Shot, Few-Shot und 2-Stage CoT über die 8 Kern-Produktfamilien.



In [4]:
core_8_fams = [
    'Suprasorb P (Schaumstoff)',
    'Solvaline / Lomatuell (Atraumatische Auflage)',
    'Suprasorb X (Hydrobalance)',
    'Suprasorb Liquacel (Hydrofiber)',
    'Suprasorb A (Alginat)',
    'Vliwasorb (Superabsorber)',
    'Suprasorb G (Gel)',
    'Vliwaktiv (Aktivkohle)'
]

c_zero = get_fam_counts(df_zero, 'praeferenz_wundauflage')
c_few = get_fam_counts(df_few, 'praeferenz_wundauflage')
c_two = get_fam_counts(df_two, 'praeferenz_wundauflage')

dist_rows = []
for fam in core_8_fams:
    cnt_gt1 = c_gt1.get(fam, 0)
    cnt_gt2 = c_gt2.get(fam, 0)
    cnt_z = c_zero.get(fam, 0)
    cnt_f = c_few.get(fam, 0)
    cnt_t = c_two.get(fam, 0)
    
    dist_rows.append({
        'Produktfamilie (Wirkprinzip)': fam,
        'GT 1': f"{cnt_gt1} ({cnt_gt1/58*100:.1f}%)",
        'GT 2': f"{cnt_gt2} ({cnt_gt2/58*100:.1f}%)",
        'Zero-Shot': f"{cnt_z} ({cnt_z/58*100:.1f}%)",
        'Few-Shot': f"{cnt_f} ({cnt_f/58*100:.1f}%)",
        '2-Stage CoT': f"{cnt_t} ({cnt_t/58*100:.1f}%)"
    })

df_dist = pd.DataFrame(dist_rows)
display(df_dist)



,Produktfamilie (Wirkprinzip),GT 1,GT 2,Zero-Shot,Few-Shot,2-Stage CoT
0,Suprasorb P (Schaumstoff),24 (41.4%),30 (51.7%),35 (60.3%),30 (51.7%),43 (74.1%)
1,Solvaline / Lomatuell (Atraumatische Auflage),12 (20.7%),27 (46.6%),2 (3.4%),4 (6.9%),2 (3.4%)
2,Suprasorb X (Hydrobalance),9 (15.5%),27 (46.6%),6 (10.3%),10 (17.2%),9 (15.5%)
3,Suprasorb Liquacel (Hydrofiber),7 (12.1%),9 (15.5%),20 (34.5%),11 (19.0%),25 (43.1%)
4,Suprasorb A (Alginat),0 (0.0%),2 (3.4%),21 (36.2%),18 (31.0%),14 (24.1%)
5,Vliwasorb (Superabsorber),0 (0.0%),5 (8.6%),13 (22.4%),15 (25.9%),5 (8.6%)
6,Suprasorb G (Gel),12 (20.7%),6 (10.3%),7 (12.1%),7 (12.1%),6 (10.3%)
7,Vliwaktiv (Aktivkohle),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


## 4. Delta Modell minus Majority-Baseline auf gruppierter Ebene
Differenz $\Delta = \text{Score}_{\text{Modell}} - \text{Score}_{\text{Majority}}$ (in Prozentpunkten) je Experte.



In [5]:
maj_f1_e1, _ = eval_model(None, None, None, df_gt1)
maj_f1_e2, _ = eval_model(None, None, None, df_gt2)

delta_rows = []
for name, df_m, col_p, col_a in [
    ('Zero-Shot', df_zero, 'praeferenz_wundauflage', 'alternativ_wundauflage'),
    ('Few-Shot', df_few, 'praeferenz_wundauflage', 'alternativ_wundauflage'),
    ('2-Stage CoT', df_two, 'praeferenz_wundauflage', 'alternativ_wundauflage')
]:
    f1_e1, _ = eval_model(df_m, col_p, col_a, df_gt1)
    f1_e2, _ = eval_model(df_m, col_p, col_a, df_gt2)
    
    d_e1 = (f1_e1 - maj_f1_e1) * 100
    d_e2 = (f1_e2 - maj_f1_e2) * 100
    
    delta_rows.append({
        'Modell': name,
        'Score Exp 1': f"{f1_e1*100:.2f}%",
        'Majority Exp 1': f"{maj_f1_e1*100:.2f}%",
        'Δ Exp 1 (Prozentpunkte)': f"{d_e1:+.2f} pp",
        'Score Exp 2': f"{f1_e2*100:.2f}%",
        'Majority Exp 2': f"{maj_f1_e2*100:.2f}%",
        'Δ Exp 2 (Prozentpunkte)': f"{d_e2:+.2f} pp"
    })

df_delta = pd.DataFrame(delta_rows)
display(df_delta)



,Modell,Score Exp 1,Majority Exp 1,Δ Exp 1 (Prozentpunkte),Score Exp 2,Majority Exp 2,Δ Exp 2 (Prozentpunkte)
0,Zero-Shot,35.34%,41.09%,-5.75 pp,57.99%,56.03%,+1.95 pp
1,Few-Shot,38.74%,41.09%,-2.36 pp,60.11%,56.03%,+4.08 pp
2,2-Stage CoT,45.11%,41.09%,+4.02 pp,65.17%,56.03%,+9.14 pp


## 5. Anteil der Modellempfehlungen außerhalb der 8 Kernfamilien
Anteil und Aufschlüsselung der Empfehlungen, die nicht zu den 8 Kern-Produktfamilien gehören.



In [6]:
core_set = set(core_8_fams)
out_rows = []

for name, df_m in [('Zero-Shot', df_zero), ('Few-Shot', df_few), ('2-Stage CoT', df_two)]:
    total_preds = 0
    outside_preds = 0
    outside_counter = Counter()
    
    for img_id in common_58:
        row = df_m[df_m['image_id'] == img_id].iloc[0]
        val_p = row.get('praeferenz_wundauflage')
        mapped_set = map_product_set(to_clean_set(val_p), group_by_family=True)
        
        for item in mapped_set:
            total_preds += 1
            if item not in core_set:
                outside_preds += 1
                outside_counter[item] += 1
                
    out_rows.append({
        'Modell': name,
        'Gesamt Empfehlungen': total_preds,
        'Außerhalb Kernfamilien': outside_preds,
        'Anteil (%)': f"{outside_preds/total_preds*100:.2f}%",
        'Aufschlüsselung': dict(outside_counter)
    })

df_out = pd.DataFrame(out_rows)
display(df_out)



,Modell,Gesamt Empfehlungen,Außerhalb Kernfamilien,Anteil (%),Aufschlüsselung
0,Zero-Shot,107,3,2.80%,{'Suprasorb F (Folie)': 3}
1,Few-Shot,96,1,1.04%,{'Suprasorb F (Folie)': 1}
2,2-Stage CoT,109,5,4.59%,"{'Metalline': 1, 'Suprasorb F (Folie)': 3, 'Su..."
